# Important Insights Analysis Notebook

This notebook summarizes and visualizes the thesis-level results generated by `02ScalingPhrase/localmodel/analyze_results.py`.

It is intentionally dependency-light: it uses only the Python standard library plus IPython display utilities, so it does not require pandas, seaborn, or matplotlib.

In [1]:
from __future__ import annotations

import csv
import html
from pathlib import Path
from IPython.display import HTML, Markdown, display

def find_analysis_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / '02ScalingPhrase' / 'localmodel' / 'results' / 'analysis',
        Path('/Users/macos/LearningSpace/ue-courses/Thesis/02ScalingPhrase/localmodel/results/analysis'),
    ]
    for candidate in candidates:
        if (candidate / 'summary_by_kind.csv').exists():
            return candidate
    raise FileNotFoundError('Could not find analysis CSV files. Run analyze_results.py first.')

ANALYSIS_DIR = find_analysis_dir()
display(Markdown(f'Using analysis directory: `{ANALYSIS_DIR}`'))

def load_csv(name: str) -> list[dict]:
    path = ANALYSIS_DIR / name
    with path.open('r', encoding='utf-8', newline='') as handle:
        return list(csv.DictReader(handle))

def fnum(value, default=0.0):
    try:
        if value in ('', None):
            return default
        return float(value)
    except (TypeError, ValueError):
        return default

def pct(value):
    return f'{fnum(value) * 100:.1f}%'

def pp(value):
    return f'{fnum(value):+.1f} pp'

def fmt(value, kind='auto'):
    if kind == 'pct':
        return pct(value)
    if kind == 'pp':
        return pp(value)
    if isinstance(value, str):
        return html.escape(value)
    return html.escape(str(value))

def table(rows, columns, formats=None, max_rows=20):
    formats = formats or {}
    visible = rows[:max_rows]
    head = ''.join(f'<th>{html.escape(col)}</th>' for col in columns)
    body = []
    for row in visible:
        cells = ''.join(f'<td>{fmt(row.get(col, ""), formats.get(col, "auto"))}</td>' for col in columns)
        body.append(f'<tr>{cells}</tr>')
    note = f'<p><em>Showing {len(visible)} of {len(rows)} rows.</em></p>' if len(rows) > len(visible) else ''
    css = '<style>table.insight{border-collapse:collapse;font-size:13px} .insight th,.insight td{border:1px solid #ddd;padding:5px 7px;text-align:left}.insight th{background:#f4f4f4}.insight tr:nth-child(even){background:#fafafa}</style>'
    display(HTML(css + f'<table class="insight"><thead><tr>{head}</tr></thead><tbody>{"".join(body)}</tbody></table>{note}'))

def shorten_label(text: object, max_chars: int = 38) -> str:
    text = str(text)
    return text if len(text) <= max_chars else text[: max_chars - 1] + '…'

BAR_CHART_COUNTER = 0

def bar_chart(rows, label_key, value_key, title, value_format='pct', width=1040, height=None, color='#2f6f73', left=360, right=140):
    global BAR_CHART_COUNTER
    BAR_CHART_COUNTER += 1
    clip_id = f'labelClip{BAR_CHART_COUNTER}'
    rows = list(rows)
    if not rows:
        display(Markdown('_No rows to plot._'))
        return
    height = height or max(280, 58 + len(rows) * 34)
    top, bottom = 64, 32
    inner_w = width - left - right
    bar_h = max(15, (height - top - bottom) / len(rows) - 8)
    values = [fnum(r[value_key]) for r in rows]
    if value_format == 'pct':
        value_labels = [pct(v) for v in values]
    elif value_format == 'pp':
        value_labels = [pp(v) for v in values]
    else:
        value_labels = [f'{v:.2f}' for v in values]
    min_v, max_v = min(values), max(values)
    span = max(max_v - min_v, 0.001)
    zero_x = left if min_v >= 0 else left + inner_w * ((0 - min_v) / span)
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#fbfaf7"/>',
        f'<text x="{left}" y="34" font-family="Arial" font-size="20" font-weight="700">{html.escape(title)}</text>',
        f'<clipPath id="{clip_id}"><rect x="0" y="{top-12}" width="{left-18}" height="{height-top-bottom+24}"/></clipPath>',
    ]
    if min_v < 0 < max_v:
        parts.append(f'<line x1="{zero_x:.1f}" x2="{zero_x:.1f}" y1="{top-8}" y2="{height-bottom}" stroke="#555" stroke-width="1"/>')
    for i, row in enumerate(rows):
        y = top + i * (bar_h + 8)
        label = html.escape(shorten_label(row[label_key]))
        full_label = html.escape(str(row[label_key]))
        value = values[i]
        if min_v < 0:
            x0 = left + inner_w * ((min(value, 0) - min_v) / span)
            x1 = left + inner_w * ((max(value, 0) - min_v) / span)
            bar_w = max(1, x1 - x0)
        else:
            x0 = left
            bar_w = max(1, inner_w * (value / max(max_v, 0.001)))
        parts.append(f'<text clip-path="url(#{clip_id})" x="{left-28}" y="{y + bar_h*.68:.1f}" text-anchor="end" font-family="Arial" font-size="12"><title>{full_label}</title>{label}</text>')
        parts.append(f'<rect x="{x0:.1f}" y="{y:.1f}" width="{bar_w:.1f}" height="{bar_h:.1f}" rx="3" fill="{color}"/>')
        if value < 0:
            text_x = x0 + 10
            anchor = 'start'
            text_fill = '#ffffff' if bar_w >= 55 else '#111111'
        else:
            text_x = min(width - 8, x0 + bar_w + 10)
            anchor = 'start'
            text_fill = '#111111'
        parts.append(f'<text x="{text_x:.1f}" y="{y + bar_h*.68:.1f}" text-anchor="{anchor}" font-family="Arial" font-size="12" fill="{text_fill}">{value_labels[i]}</text>')
    parts.append('</svg>')
    display(HTML(''.join(parts)))

Using analysis directory: `/Users/macos/LearningSpace/ue-courses/Thesis/02ScalingPhrase/localmodel/results/analysis`

In [2]:
summary_by_kind = load_csv('summary_by_kind.csv')
summary_by_model_kind = load_csv('summary_by_model_and_kind.csv')
prompt_contrasts = load_csv('prompt_factor_contrasts.csv')
heterogeneity = load_csv('prompt_factor_heterogeneity.csv')
kind_differences = load_csv('model_kind_factor_differences.csv')
behavior = load_csv('behavior_summary.csv')
errors = load_csv('error_category_summary.csv')
dataset_shift = load_csv('dataset_shift_summary.csv')
paired = load_csv('paired_fixed_effect_contrasts.csv')
model_sizes = load_csv('model_size_comparison.csv') if (ANALYSIS_DIR / 'model_size_comparison.csv').exists() else []

display(Markdown('## Loaded artifacts'))
table([
    {'artifact': 'summary_by_kind.csv', 'rows': len(summary_by_kind)},
    {'artifact': 'summary_by_model_and_kind.csv', 'rows': len(summary_by_model_kind)},
    {'artifact': 'prompt_factor_contrasts.csv', 'rows': len(prompt_contrasts)},
    {'artifact': 'prompt_factor_heterogeneity.csv', 'rows': len(heterogeneity)},
    {'artifact': 'model_kind_factor_differences.csv', 'rows': len(kind_differences)},
    {'artifact': 'behavior_summary.csv', 'rows': len(behavior)},
    {'artifact': 'error_category_summary.csv', 'rows': len(errors)},
    {'artifact': 'dataset_shift_summary.csv', 'rows': len(dataset_shift)},
], ['artifact', 'rows'])

## Loaded artifacts

artifact,rows
summary_by_kind.csv,2
summary_by_model_and_kind.csv,12
prompt_factor_contrasts.csv,70
prompt_factor_heterogeneity.csv,10
model_kind_factor_differences.csv,30
behavior_summary.csv,102
error_category_summary.csv,75
dataset_shift_summary.csv,43


## 0. Model Size Comparison

All size values below are taken from local Ollama manifests and reported only in GiB. The GiB value is the quantized model package/model layer size, not FP16 memory size.

In [3]:
if model_sizes:
    size_rows = sorted(model_sizes, key=lambda r: fnum(r['nominal_parameters_b']))
    table(
        size_rows,
        ['model_tag', 'nominal_parameters_b', 'local_or_registry_gib'],
        max_rows=10,
    )
    for row in size_rows:
        row['label'] = row['model_tag']
    bar_chart(size_rows, 'label', 'local_or_registry_gib', 'Local Ollama model size in GiB', value_format='number', height=380, color='#7c6fbd')
else:
    display(Markdown('_No model_size_comparison.csv found._'))


model_tag,nominal_parameters_b,local_or_registry_gib
nemotron-3-nano:4b,3.97,2.643
deepseek-coder:6.7b,6.74,3.565
qwen2.5:7b,7,4.361
mistral:7b-instruct,7.24,4.073
llama3.1:8b,8,4.583
gemma2:9b,9,5.069


## 1. Task Kind Is The First Split

The scaled study should not be interpreted as a single pooled result. Code and math have different baselines and different sensitivity to prompt factors.

In [4]:
table(summary_by_kind, ['kind', 'n', 'passed', 'pass_rate', 'mean_score', 'median_duration_seconds', 'mean_prompt_tokens', 'mean_completion_tokens'], {'pass_rate': 'pct'})
bar_chart(summary_by_kind, 'kind', 'pass_rate', 'Pass rate by task kind')

kind,n,passed,pass_rate,mean_score,median_duration_seconds,mean_prompt_tokens,mean_completion_tokens
code,5400,492,9.1%,0.1374074074074074,9.563375234603882,291.0118518518519,564.7614814814815
math,5400,1679,31.1%,0.31092592592592594,1.8313188552856445,163.67592592592592,156.5940740740741


## 2. Model Ranking Depends On Task Kind

This chart makes the main source of distribution shift visible: adding math and adding models changes the experimental population, not only the sample size.

In [5]:
rows = sorted(summary_by_model_kind, key=lambda r: (r['kind'], -fnum(r['pass_rate'])))
for r in rows:
    r['label'] = f"{r['kind']} / {r['model_slug'].replace('_1_50', '')}"
bar_chart(rows, 'label', 'pass_rate', 'Pass rate by model and task kind', height=520, color='#3f7fbf')
table(rows, ['kind', 'model_slug', 'n', 'passed', 'pass_rate', 'mean_score', 'median_duration_seconds'], {'pass_rate': 'pct'}, max_rows=12)

kind,model_slug,n,passed,pass_rate,mean_score,median_duration_seconds
code,nemotron-3-nano-4b_1_50,900,205,22.8%,0.30396825396825394,17.78443205356598
code,qwen2.5-7b_1_50,900,119,13.2%,0.17269841269841268,8.754344701766968
code,gemma2-9b_1_50,900,102,11.3%,0.19428571428571428,5.524169445037842
code,llama3.1-8b_1_50,900,52,5.8%,0.10873015873015873,11.161959886550903
code,deepseek-coder-6.7b-instruct_1_50,900,10,1.1%,0.020158730158730157,9.041467666625977
code,mistral-7b-instruct_1_50,900,4,0.4%,0.024603174603174603,9.120986580848694
math,nemotron-3-nano-4b_1_50,900,821,91.2%,0.9122222222222223,3.1529808044433594
math,llama3.1-8b_1_50,900,389,43.2%,0.43222222222222223,5.085116624832153
math,qwen2.5-7b_1_50,900,212,23.6%,0.23555555555555555,0.6785002946853638
math,gemma2-9b_1_50,900,137,15.2%,0.15222222222222223,0.8019598722457886


## 3. Prompt Factor Main Effects And Fixed-Effect-Style Contrasts

The first table is the raw factor contrast. The second table compares factor levels within the same model, task kind, and base task, which is closer to a fixed-effect interpretation.

In [6]:
overall_contrasts = [r for r in prompt_contrasts if not r.get('model_slug')]
overall_contrasts = sorted(overall_contrasts, key=lambda r: (r['kind'], r['factor'], -abs(fnum(r['delta_pass_rate_pp']))))
for r in overall_contrasts:
    r['label'] = f"{r['kind']} / {r['factor']}={r['level']}"
bar_chart(overall_contrasts, 'label', 'delta_pass_rate_pp', 'Raw factor deltas versus baseline', value_format='pp', height=390, color='#7c6fbd')
table(overall_contrasts, ['kind', 'factor', 'baseline', 'level', 'baseline_pass_rate', 'level_pass_rate', 'delta_pass_rate_pp', 'z_test_p_value'], {'baseline_pass_rate': 'pct', 'level_pass_rate': 'pct', 'delta_pass_rate_pp': 'pp'}, max_rows=12)

fixed_rows = [r for r in paired if r.get('model_slug') == 'ALL_MODELS_FIXED_BY_TASK']
for r in fixed_rows:
    r['label'] = f"{r['kind']} / {r['factor']}={r['level']}"
bar_chart(fixed_rows, 'label', 'mean_pass_delta_pp', 'Paired fixed-effect-style pass deltas', value_format='pp', height=390, color='#c75d4c')
table(fixed_rows, ['kind', 'factor', 'baseline', 'level', 'paired_units', 'mean_score_delta', 'mean_pass_delta_pp', 'positive_score_units', 'negative_score_units', 'zero_score_units'], {'mean_pass_delta_pp': 'pp'}, max_rows=12)

kind,factor,baseline,level,baseline_pass_rate,level_pass_rate,delta_pass_rate_pp,z_test_p_value
code,format,plaintext,xml,8.8%,9.6%,+0.8 pp,0.386924060376156
code,format,plaintext,json,8.8%,8.9%,+0.2 pp,0.8603380688997779
code,role_binary,no role,role,9.6%,8.6%,-1.0 pp,0.18547054704628943
code,tone,neutral,polite,9.3%,8.8%,-0.5 pp,0.6006852677074704
code,tone,neutral,aggressive,9.3%,9.3%,+0.0 pp,1.0
math,format,plaintext,json,33.2%,29.1%,-4.2 pp,0.006945917837891586
math,format,plaintext,xml,33.2%,31.0%,-2.2 pp,0.15333585361754165
math,role_binary,no role,role,30.0%,32.2%,+2.2 pp,0.08281566776640613
math,tone,neutral,polite,30.2%,31.9%,+1.7 pp,0.27989085966240845
math,tone,neutral,aggressive,30.2%,31.2%,+0.9 pp,0.5390140723788477


kind,factor,baseline,level,paired_units,mean_score_delta,mean_pass_delta_pp,positive_score_units,negative_score_units,zero_score_units
code,format,plaintext,json,1800,0.008095238095238095,+0.0 pp,174,108,1518
code,format,plaintext,xml,1800,0.004285714285714286,+0.3 pp,144,132,1524
code,role_binary,no role,role,2700,-0.03761904761904762,-3.0 pp,216,288,2196
code,tone,neutral,aggressive,1800,0.0028571428571428567,+1.0 pp,150,138,1512
code,tone,neutral,polite,1800,0.03809523809523809,+3.7 pp,204,138,1458
math,format,plaintext,json,1800,-0.016666666666666666,-1.7 pp,78,108,1614
math,format,plaintext,xml,1800,0.0033333333333333335,+0.3 pp,96,90,1614
math,role_binary,no role,role,2700,0.03666666666666667,+3.7 pp,234,135,2331
math,tone,neutral,aggressive,1800,0.006666666666666667,+0.7 pp,102,90,1608
math,tone,neutral,polite,1800,-0.016666666666666666,-1.7 pp,42,72,1686


## 4. Model-Specific Heterogeneity

A factor is not necessarily good or bad globally. The same factor can help one model and hurt another.

In [7]:
het = sorted(heterogeneity, key=lambda r: -fnum(r['range_delta_pp']))
for r in het:
    r['label'] = f"{r['kind']} / {r['factor']}={r['level']}"
bar_chart(het[:10], 'label', 'range_delta_pp', 'Model disagreement range for each factor level', value_format='pp', height=390, color='#d89b34')
table(het, ['kind', 'factor', 'level', 'mean_delta_pp', 'min_delta_pp', 'min_model', 'max_delta_pp', 'max_model', 'range_delta_pp', 'positive_models', 'negative_models'], {'mean_delta_pp': 'pp', 'min_delta_pp': 'pp', 'max_delta_pp': 'pp', 'range_delta_pp': 'pp'}, max_rows=12)

kind,factor,level,mean_delta_pp,min_delta_pp,min_model,max_delta_pp,max_model,range_delta_pp,positive_models,negative_models
math,format,json,-4.2 pp,-15.7 pp,llama3.1-8b_1_50,+1.3 pp,nemotron-3-nano-4b_1_50,+17.0 pp,1,4
math,format,xml,-2.2 pp,-9.7 pp,llama3.1-8b_1_50,+5.7 pp,qwen2.5-7b_1_50,+15.3 pp,2,4
math,role_binary,role,+2.2 pp,-0.7 pp,deepseek-coder-6.7b-instruct_1_50,+10.2 pp,qwen2.5-7b_1_50,+10.9 pp,5,1
math,tone,polite,+1.7 pp,-1.7 pp,nemotron-3-nano-4b_1_50,+5.3 pp,llama3.1-8b_1_50,+7.0 pp,4,2
code,tone,aggressive,+0.0 pp,-2.0 pp,llama3.1-8b_1_50,+2.0 pp,deepseek-coder-6.7b-instruct_1_50,+4.0 pp,4,2
code,format,xml,+0.8 pp,+0.0 pp,gemma2-9b_1_50,+3.7 pp,nemotron-3-nano-4b_1_50,+3.7 pp,4,0
math,tone,aggressive,+0.9 pp,-0.7 pp,nemotron-3-nano-4b_1_50,+2.3 pp,deepseek-coder-6.7b-instruct_1_50,+3.0 pp,4,2
code,tone,polite,-0.5 pp,-2.7 pp,llama3.1-8b_1_50,+0.3 pp,deepseek-coder-6.7b-instruct_1_50,+3.0 pp,2,2
code,role_binary,role,-1.0 pp,-2.2 pp,gemma2-9b_1_50,+0.4 pp,llama3.1-8b_1_50,+2.7 pp,1,5
code,format,json,+0.2 pp,-1.0 pp,gemma2-9b_1_50,+1.0 pp,deepseek-coder-6.7b-instruct_1_50,+2.0 pp,3,2


## 5. Math Versus Code Factor Behavior

This is the most important interaction view for the scaled experiment. It shows whether the same prompt factor behaves differently for math and code within the same model.

In [8]:
kd = sorted(kind_differences, key=lambda r: -abs(fnum(r['math_minus_code_delta_pp'])))
for r in kd:
    short = r['model_slug'].replace('_1_50', '')
    r['label'] = f"{short} / {r['factor']}={r['level']}"

bar_chart(kd[:15], 'label', 'math_minus_code_delta_pp', 'How much more a factor helps math than code', value_format='pp', height=600, color='#2f6f73')
table(
    kd,
    ['model_slug', 'factor', 'level', 'code_delta_pp', 'math_delta_pp', 'math_minus_code_delta_pp',
     'code_delta_prompt_tokens', 'math_delta_prompt_tokens', 'math_minus_code_prompt_tokens',
     'code_delta_completion_tokens', 'math_delta_completion_tokens', 'math_minus_code_completion_tokens',
     'code_delta_duration_seconds', 'math_delta_duration_seconds', 'math_minus_code_duration_seconds',
     'code_p_value', 'math_p_value'],
    {'code_delta_pp': 'pp', 'math_delta_pp': 'pp', 'math_minus_code_delta_pp': 'pp'},
    max_rows=20,
)

# Additional visual diagnostics: prompt cost, generated-token cost, and latency.
for metric, title, color in [
    ('math_minus_code_prompt_tokens', 'Math-vs-code difference in prompt-token effect', '#7c6fbd'),
    ('math_minus_code_completion_tokens', 'Math-vs-code difference in completion-token effect', '#c75d4c'),
    ('math_minus_code_duration_seconds', 'Math-vs-code difference in duration effect', '#d89b34'),
]:
    rows = sorted(kind_differences, key=lambda r: -abs(fnum(r.get(metric))))
    for r in rows:
        short = r['model_slug'].replace('_1_50', '')
        r['label'] = f"{short} / {r['factor']}={r['level']}"
    bar_chart(rows[:15], 'label', metric, title, value_format='number', height=600, color=color)


model_slug,factor,level,code_delta_pp,math_delta_pp,math_minus_code_delta_pp,code_delta_prompt_tokens,math_delta_prompt_tokens,math_minus_code_prompt_tokens,code_delta_completion_tokens,math_delta_completion_tokens,math_minus_code_completion_tokens,code_delta_duration_seconds,math_delta_duration_seconds,math_minus_code_duration_seconds,code_p_value,math_p_value
llama3.1-8b_1_50,format,json,-0.3 pp,-15.7 pp,-15.3 pp,16.079999999999984,12.699999999999989,-3.3799999999999955,-19.103333333333353,-129.5666666666667,-110.46333333333334,-0.5207394814491284,-3.7084205587704977,-3.1876810773213693,0.8578813481418388,0.0001101563463264192
qwen2.5-7b_1_50,role_binary,role,-1.1 pp,+10.2 pp,+11.3 pp,3.0,2.0,-1.0,23.555555555555543,2.297777777777778,-21.257777777777765,0.6358171913358888,0.055737808015611456,-0.5800793833202773,0.6226975927106597,0.0003021992422249198
llama3.1-8b_1_50,format,xml,+0.7 pp,-9.7 pp,-10.3 pp,21.30000000000001,16.25999999999999,-5.0400000000000205,-12.160000000000025,16.563333333333333,28.723333333333358,-0.3310737490653999,0.4881634847323095,0.8192372337977094,0.7309913738232376,0.017663225455655202
llama3.1-8b_1_50,tone,polite,-2.7 pp,+5.3 pp,+8.0 pp,29.00000000000003,32.0,2.9999999999999716,60.53666666666669,45.60333333333335,-14.933333333333337,1.7370652540524816,1.3473470218976331,-0.38971823215484847,0.16906066842595452,0.18783777367488144
deepseek-coder-6.7b-instruct_1_50,format,xml,+0.3 pp,-5.7 pp,-6.0 pp,33.71999999999997,28.579999999999984,-5.139999999999986,-8.386666666666656,-4.263333333333321,4.123333333333335,-0.08810582955678292,-0.6046474067370102,-0.5165415771802273,0.6533676621999003,0.003912182095129829
qwen2.5-7b_1_50,format,xml,+0.0 pp,+5.7 pp,+5.7 pp,21.30000000000001,16.26000000000002,-5.039999999999992,14.453333333333319,1.1499999999999995,-13.30333333333332,0.3878393499056507,0.03554959138234459,-0.35228975852330613,1.0,0.10819462291873343
deepseek-coder-6.7b-instruct_1_50,format,json,+1.0 pp,-4.3 pp,-5.3 pp,18.51333333333332,15.873333333333335,-2.6399999999999864,-23.95333333333332,-22.839999999999975,1.113333333333344,-0.43325722853342796,-0.9815814963976548,-0.5483242678642268,0.2540504721886064,0.035431124882258754
nemotron-3-nano-4b_1_50,format,xml,+3.7 pp,-0.7 pp,-4.3 pp,24.933333333333337,18.600000000000023,-6.333333333333314,-64.1533333333332,-1.5166666666666515,62.63666666666654,-1.3598875276247675,-0.014342962106068669,1.3455445655186988,0.28714133928909275,0.7789555081438064
deepseek-coder-6.7b-instruct_1_50,tone,polite,+0.3 pp,+4.7 pp,+4.3 pp,30.333333333333314,31.333333333333343,1.0000000000000284,12.99333333333334,24.883333333333383,11.890000000000043,0.2884376025199895,1.069819980462392,0.7813823779424025,0.5627247518718889,0.013434342905140745
gemma2-9b_1_50,format,json,-1.0 pp,-5.3 pp,-4.3 pp,18.0,14.560000000000002,-3.4399999999999977,-4.653333333333336,-0.9066666666666672,3.746666666666669,-0.16827779054641745,-0.020392334461212136,0.1478854560852053,0.6973779896044128,0.07233013782293779


## 6. Behavior And Error Modes

This section checks whether prompt framing changes model behavior: verbosity, latency, code-only compliance, extraction failure, and runtime failure.

In [9]:
behavior_main = [r for r in behavior if r.get('grouping') in {'kind', 'kind+tone', 'kind+format'}]
table(behavior_main, ['grouping', 'kind', 'tone', 'format', 'n', 'pass_rate', 'mean_score', 'mean_completion_tokens', 'mean_duration_seconds', 'code_only_rate', 'extraction_failure_rate', 'runtime_failure_rate'], {'pass_rate': 'pct', 'code_only_rate': 'pct', 'extraction_failure_rate': 'pct', 'runtime_failure_rate': 'pct'}, max_rows=30)

error_main = [r for r in errors if r.get('grouping') == 'kind+error_category' and r.get('error_category') != 'none']
for r in error_main:
    r['label'] = f"{r['kind']} / {r['error_category']}"
bar_chart(error_main, 'label', 'rate_within_group', 'Error category rate by task kind', height=300, color='#67717a')
table(error_main, ['kind', 'error_category', 'n', 'rate_within_group'], {'rate_within_group': 'pct'})

grouping,kind,tone,format,n,pass_rate,mean_score,mean_completion_tokens,mean_duration_seconds,code_only_rate,extraction_failure_rate,runtime_failure_rate
kind,code,,,5400,9.1%,0.1374074074074074,564.7614814814815,12.243028557786236,72.9%,0.3%,54.7%
kind,math,,,5400,31.1%,0.31092592592592594,156.5940740740741,3.887234082840107,0.0%,0.0%,0.0%
kind+tone,code,aggressive,,1800,9.3%,0.1392857142857143,557.5027777777777,11.987996952401268,74.3%,0.3%,53.4%
kind+tone,code,neutral,,1800,9.3%,0.1411111111111111,553.5688888888889,11.88971587393019,75.7%,0.4%,53.0%
kind+tone,code,polite,,1800,8.8%,0.13182539682539682,583.2127777777778,12.85137284702725,68.7%,0.1%,57.8%
kind+tone,math,aggressive,,1800,31.2%,0.31166666666666665,155.67555555555555,3.8958776387903424,0.0%,0.0%,0.0%
kind+tone,math,neutral,,1800,30.2%,0.3022222222222222,148.73555555555555,3.6064545028739508,0.0%,0.0%,0.0%
kind+tone,math,polite,,1800,31.9%,0.3188888888888889,165.3711111111111,4.159370106856028,0.0%,0.0%,0.0%
kind+format,code,,json,1800,8.9%,0.13682539682539682,561.4483333333334,12.13796780122651,73.5%,0.2%,52.9%
kind+format,code,,plaintext,1800,8.8%,0.1338095238095238,576.9627777777778,12.526246152321498,68.9%,0.3%,56.9%


kind,error_category,n,rate_within_group
code,invalid_python,207,3.8%
code,missing_or_empty_code,14,0.3%
code,name_not_defined,2674,49.5%
code,timeout,61,1.1%
code,wrong_answer,390,7.2%
math,wrong_answer,3721,68.9%


## 7. Dataset Shift: Pilot Versus Scaled Study

`scaled_common3_code` is the closest comparison to the original pilot: the same three model families and code-only rows. `scaled_all` includes the added models and math tasks.

In [10]:
shift_overall = [r for r in dataset_shift if r.get('grouping') in {'dataset', 'dataset+kind'}]
for r in shift_overall:
    r['label'] = f"{r['dataset']}" + (f" / {r['kind']}" if r.get('kind') else '')
bar_chart(shift_overall, 'label', 'mean_score', 'Mean score under pilot versus scaled populations', height=310, color='#3f7fbf')
table(shift_overall, ['grouping', 'dataset', 'kind', 'n', 'models', 'base_tasks', 'pass_rate', 'mean_score', 'mean_completion_tokens', 'mean_duration_seconds', 'code_only_rate', 'runtime_failure_rate'], {'pass_rate': 'pct', 'code_only_rate': 'pct', 'runtime_failure_rate': 'pct'}, max_rows=12)

shift_tone = [r for r in dataset_shift if r.get('grouping') == 'dataset+kind+tone']
table(shift_tone, ['dataset', 'kind', 'tone', 'n', 'pass_rate', 'mean_score', 'mean_completion_tokens', 'runtime_failure_rate'], {'pass_rate': 'pct', 'runtime_failure_rate': 'pct'}, max_rows=18)

grouping,dataset,kind,n,models,base_tasks,pass_rate,mean_score,mean_completion_tokens,mean_duration_seconds,code_only_rate,runtime_failure_rate
dataset,pilot_codeonly,,2700,3,50,10.3%,0.15714285714285714,289.57444444444445,9.004672222490663,99.5%,39.9%
dataset,scaled_all,,10800,6,100,20.1%,0.22416666666666665,360.6777777777778,8.06513132031317,36.4%,27.4%
dataset,scaled_common3_code,,2700,3,50,10.1%,0.15857142857142856,295.0485185185185,9.022627111187688,99.6%,40.4%
dataset+kind,pilot_codeonly,code,2700,3,50,10.3%,0.15714285714285714,289.57444444444445,9.004672222490663,99.5%,39.9%
dataset+kind,scaled_all,code,5400,6,50,9.1%,0.1374074074074074,564.7614814814815,12.243028557786236,72.9%,54.7%
dataset+kind,scaled_all,math,5400,6,50,31.1%,0.31092592592592594,156.5940740740741,3.887234082840107,0.0%,0.0%
dataset+kind,scaled_common3_code,code,2700,3,50,10.1%,0.15857142857142856,295.0485185185185,9.022627111187688,99.6%,40.4%


dataset,kind,tone,n,pass_rate,mean_score,mean_completion_tokens,runtime_failure_rate
pilot_codeonly,code,aggressive,900,9.2%,0.14412698412698413,280.83222222222224,39.4%
pilot_codeonly,code,neutral,900,12.3%,0.18206349206349207,260.35333333333335,33.4%
pilot_codeonly,code,polite,900,9.2%,0.14523809523809525,327.53777777777776,46.9%
scaled_all,code,aggressive,1800,9.3%,0.1392857142857143,557.5027777777777,53.4%
scaled_all,code,neutral,1800,9.3%,0.1411111111111111,553.5688888888889,53.0%
scaled_all,code,polite,1800,8.8%,0.13182539682539682,583.2127777777778,57.8%
scaled_all,math,aggressive,1800,31.2%,0.31166666666666665,155.67555555555555,0.0%
scaled_all,math,neutral,1800,30.2%,0.3022222222222222,148.73555555555555,0.0%
scaled_all,math,polite,1800,31.9%,0.3188888888888889,165.3711111111111,0.0%
scaled_common3_code,code,aggressive,900,9.8%,0.15444444444444444,285.52222222222224,40.1%


## 8. Model Size Context

Use this as context only. Parameter count and quantized GiB size are not performance controls by themselves, but they help explain why comparing models as if they were identical is risky.

In [11]:
if model_sizes:
    size_rows = sorted(model_sizes, key=lambda r: fnum(r['nominal_parameters_b']))
    for r in size_rows:
        r['label'] = r['model_tag']
    bar_chart(size_rows, 'label', 'local_or_registry_gib', 'Quantized model package size in GiB', value_format='number', height=360, color='#7c6fbd')
    table(size_rows, ['model_tag', 'nominal_parameters_b', 'local_or_registry_gib', 'size_source', 'notes'], max_rows=10)
else:
    display(Markdown('_No model_size_comparison.csv found._'))

model_tag,nominal_parameters_b,local_or_registry_gib,size_source,notes
nemotron-3-nano:4b,3.97,2.643,local Ollama manifest,Measured from local manifest model layer size: registry.ollama.ai/library/nemotron-3-nano/4b.
deepseek-coder:6.7b,6.74,3.565,local Ollama manifest,Measured from local manifest model layer size: registry.ollama.ai/library/deepseek-coder/6.7b.
qwen2.5:7b,7,4.361,local Ollama manifest,Measured from local manifest model layer size: registry.ollama.ai/library/qwen2.5/7b.
mistral:7b-instruct,7.24,4.073,local Ollama manifest,Measured from local manifest model layer size: registry.ollama.ai/library/mistral/7b-instruct.
llama3.1:8b,8,4.583,local Ollama manifest,Measured from local manifest model layer size: registry.ollama.ai/library/llama3.1/8b.
gemma2:9b,9,5.069,local Ollama manifest,Measured from local manifest model layer size: registry.ollama.ai/library/gemma2/9b.


## Thesis-Level Interpretation Guardrails

- Do not claim that a tone is universally better. Report effects conditional on model and task kind.
- Separate raw pass-rate contrasts from paired/fixed-effect-style contrasts.
- Treat math and code as different task domains.
- Explain the changed results as dataset shift: the scaled study adds new models, math tasks, different output behavior, and different model ability distribution.
- For the final thesis/paper, add a formal mixed-effect model if the environment supports `statsmodels` or R.